This notebook just has the creativity metric definitions outside of the demo for ease of use

This first cell has the original creativity metric used in the first demo and the presentation

In [ ]:
from typing import Union

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from sklearn.neighbors import NearestNeighbors

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

    def probs(self, x):
        return F.softmax(self.forward(x), dim=1)
# ─────────────────────────────────────────────────────────────
# 5. Creativity scoring  (Ramaswamy et al.)
# ─────────────────────────────────────────────────────────────

def score_creativity(images: torch.Tensor, clf: Classifier,
                     train_flat: np.ndarray, device: Union[str, torch.device] = "cpu",
                     alpha: float = 0.5):
    """
    Compute per-image creativity scores.

    Creativity(x) = Novelty(x)^alpha * Usefulness(x)^(1-alpha)

    Usefulness  = min(P(2|x), P(6|x))
                  Peaks when the classifier is equally confident the image
                  is a 2 AND a 6 — the ideal creative combination.

    Novelty     = nn_dist / (1 + nn_dist)
                  Saturates toward 1 as the image moves away from all
                  training examples; 0 if the image is a copy of one.
    """
    clf.eval()
    with torch.no_grad():
        probs = clf.probs(images.to(device)).cpu().numpy()   # (N, 2)

    usefulness = np.minimum(probs[:, 0], probs[:, 1])        # (N,)

    flat = images.view(images.size(0), -1).cpu().numpy()     # (N, 784)
    nbrs = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(train_flat)
    nn_dist = nbrs.kneighbors(flat)[0][:, 0]                 # (N,)
    novelty = nn_dist / (1.0 + nn_dist)

    creativity = (novelty ** alpha) * (usefulness ** (1.0 - alpha))
    return creativity, novelty, usefulness

This generates the following image: ![](./images/creative_optimised.png)

This next snippet is a more sophisticated algorithm that uses shape features to attempt to judge creativity and task accuracy

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5b. Extended creativity components
# ─────────────────────────────────────────────────────────────

# ── 5b-i. General autoencoder (trained on all MNIST) ─────────

class GeneralAE(nn.Module):
    """Simple AE trained on all 10 MNIST classes.  Low reconstruction
    error means the image looks like a plausible handwritten mark."""
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_general_ae(ae: GeneralAE, epochs: int = 10, lr: float = 1e-3,
                     device: Union[str, torch.device] = "cpu"):
    """Train the general AE on all 10 MNIST classes."""
    full_dataset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True,
        transform=transforms.ToTensor()
    )
    loader = DataLoader(full_dataset, batch_size=256, shuffle=True)
    opt = optim.Adam(ae.parameters(), lr=lr)
    ae.train()
    for epoch in range(epochs):
        total = 0.0
        for x, _ in loader:
            x = x.to(device)
            loss = F.mse_loss(ae(x), x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"  [AE] epoch {epoch+1:>3}/{epochs}  loss={total/len(loader):.6f}")


# ── 5b-ii. Persistent homology ────────────────────────────────
# Uses gudhi (cubical complex, H1) + persim (Wasserstein distance).

try:
    import gudhi
    from persim import wasserstein as _wasserstein
    TOPO_AVAILABLE = True
except ImportError:
    TOPO_AVAILABLE = False
    print("gudhi / persim not found — structural novelty will be set to 0.")
    print("Install with:  pip install gudhi persim")


def _persistence_h1(img_np: np.ndarray) -> np.ndarray:
    """H1 persistence diagram via cubical sublevel-set filtration.
    Infinite deaths are dropped; returns [[0,0]] if no finite H1 features."""
    cc = gudhi.CubicalComplex(
        dimensions=[28, 28],
        top_dimensional_cells=img_np.flatten().tolist()
    )
    cc.compute_persistence()
    dgm = cc.persistence_intervals_in_dimension(1)
    if len(dgm) == 0:
        return np.array([[0.0, 0.0]])
    finite = dgm[np.isfinite(dgm[:, 1])]
    return finite if len(finite) > 0 else np.array([[0.0, 0.0]])


def compute_train_diagrams(train_flat: np.ndarray,
                            n_samples: int = 150,
                            seed: int = 0) -> list:
    """Compute H1 diagrams for a random subset of training images."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(train_flat), size=min(n_samples, len(train_flat)),
                     replace=False)
    diagrams = []
    for i, j in enumerate(idx):
        diagrams.append(_persistence_h1(train_flat[j].reshape(28, 28)))
        if (i + 1) % 50 == 0:
            print(f"  [TOPO] {i+1}/{len(idx)} training diagrams computed")
    return diagrams

def _wd(d1, d2) -> float:
    r = _wasserstein(d1, d2, matching=False)
    return float(r[0]) if isinstance(r, tuple) else float(r)

def topological_novelty(images: torch.Tensor, train_diagrams: list) -> np.ndarray:
    """Min Wasserstein-1 distance to any training H1 diagram, normalised
    as d / (1 + d).  Requires TOPO_AVAILABLE."""
    imgs_np = images.squeeze().numpy()   # (N, 28, 28)
    scores = []
    for img in imgs_np:
        dgm = _persistence_h1(img)
        min_d = min(_wd(dgm, td) for td in train_diagrams)
        scores.append(min_d / (1.0 + min_d))
    return np.array(scores, dtype=np.float32)


# ── 5b-iii. Jensen-Shannon ambiguity ─────────────────────────


def js_ambiguity(probs: np.ndarray) -> np.ndarray:
    """
    JS divergence (log base 2) between each row of `probs` and the
    uniform distribution u = [0.5, 0.5].  Result is in [0, 1];
    equals 1 when p = [0.5, 0.5] (maximum uncertainty).
    """
    u = np.full_like(probs, 0.5)
    m = 0.5 * (probs + u)
    eps = 1e-10
    kl_pm = np.sum(probs * np.log2((probs + eps) / (m + eps)), axis=1)
    kl_um = np.sum(u     * np.log2((u     + eps) / (m + eps)), axis=1)
    return np.clip(0.5 * kl_pm + 0.5 * kl_um, 0.0, 1.0)


def js_ambiguity_torch(probs: torch.Tensor) -> torch.Tensor:
    """Differentiable version of js_ambiguity for gradient-based optimisation."""
    u = torch.full_like(probs, 0.5)
    m = 0.5 * (probs + u)
    eps = 1e-10
    kl_pm = (probs * torch.log2((probs + eps) / (m + eps))).sum(dim=1)
    kl_um = (u     * torch.log2((u     + eps) / (m + eps))).sum(dim=1)
    return torch.clamp(0.5 * kl_pm + 0.5 * kl_um, 0.0, 1.0)


# ── 5b-iv. Extended creativity score ─────────────────────────

def score_creativity_extended(images: torch.Tensor,
                               clf: Classifier,
                               ae: GeneralAE,
                               train_flat: np.ndarray,
                               train_diagrams: list,
                               device: Union[str, torch.device] = "cpu",
                               weights: tuple = (1/3, 1/3, 1/3)):
    """
    ExtCreativity = StructuralNovelty^w1 · Realness^w2 · Ambiguity^w3

    StructuralNovelty — min Wasserstein-1 distance in H1 diagram space,
                        normalised by d/(1+d).  0 if gudhi unavailable.
    Realness          — 1 / (1 + MSE(x, AE(x))); near 1 for digit-like images.
    Ambiguity         — JS divergence from classifier output to uniform [0.5, 0.5];
                        equals 1 when the classifier is maximally uncertain.
    """
    w1, w2, w3 = weights
    ae.eval()
    clf.eval()

    with torch.no_grad():
        imgs_dev = images.to(device)
        recon    = ae(imgs_dev)
        mse      = F.mse_loss(recon, imgs_dev, reduction="none") \
                     .mean(dim=[1, 2, 3]).cpu().numpy()
        probs    = clf.probs(imgs_dev).cpu().numpy()

    realness  = 1.0 / (1.0 + mse)
    ambiguity = js_ambiguity(probs)

    if TOPO_AVAILABLE and len(train_diagrams) > 0:
        struct_novelty = topological_novelty(images.cpu(), train_diagrams)
    else:
        struct_novelty = np.zeros(len(images), dtype=np.float32)

    ext_creativity = (struct_novelty ** w1) * (realness ** w2) * (ambiguity ** w3)
    return ext_creativity, struct_novelty, realness, ambiguity

This generates the following images ![](./images/creative_ext_optimised.png)

The cell below handles all of the necessary precursor steps to compute the creativity metric discussed in the slides. As we described, this metric is used for post-hoc analysis of generated images rather than as the creativity objective for a reinforcement learning algorithm, so it operated directly on generated images in data space.

In [ ]:
import os
from typing import Callable, Optional, Sequence, Union

FeatureFn = Callable[[torch.Tensor], np.ndarray]

try:
    import gudhi
    from persim import wasserstein as _wasserstein
    _TOPO_OK = True
except ImportError:
    _TOPO_OK = False

def _h1_diagram(img: np.ndarray) -> np.ndarray:
    """H1 persistence diagram (finite pairs only) of a 28x28 image via a
    cubical sublevel-set filtration.  Images are inverted so bright
    strokes form low-value basins that enclose H1 loops."""
    if not _TOPO_OK:
        raise ImportError("gudhi / persim required for H1 persistence")
    arr = 1.0 - np.asarray(img, dtype=np.float64).reshape(28, 28)
    cc = gudhi.CubicalComplex(
        dimensions=[28, 28],
        top_dimensional_cells=arr.flatten().tolist(),
    )
    cc.compute_persistence()
    dgm = cc.persistence_intervals_in_dimension(1)
    if len(dgm) == 0:
        return np.empty((0, 2), dtype=np.float64)
    finite = dgm[np.isfinite(dgm[:, 1])]
    return finite if len(finite) else np.empty((0, 2), dtype=np.float64)

def _safe_wasserstein(d1: np.ndarray, d2: np.ndarray) -> float:
    """Wasserstein-1 between two H1 diagrams, tolerant of empty diagrams
    (persim expects at least one point, so we inject a diagonal point)."""
    a = d1 if len(d1) else np.array([[0.0, 0.0]])
    b = d2 if len(d2) else np.array([[0.0, 0.0]])
    r = _wasserstein(a, b, matching=False)
    return float(r[0]) if isinstance(r, tuple) else float(r)

def h1_novelty(
    diagrams: Sequence[np.ndarray],
    reference: Sequence[np.ndarray],
) -> np.ndarray:
    """For each query diagram, minimum Wasserstein-1 distance to any
    reference diagram, squashed to [0, 1) as d / (1 + d).  High = this
    image's topology is unlike anything in the real data."""
    out = np.zeros(len(diagrams), dtype=np.float32)
    for i, dgm in enumerate(diagrams):
        d_min = min(_safe_wasserstein(dgm, ref) for ref in reference)
        out[i] = d_min / (1.0 + d_min)
    return out

def h1_summary(diagrams: Sequence[np.ndarray]) -> np.ndarray:
    """Per-image (n_features, total_persistence, max_persistence)
    summary of the H1 diagram — useful for qualitative inspection."""
    rows = []
    for dgm in diagrams:
        if len(dgm) == 0:
            rows.append((0, 0.0, 0.0))
        else:
            lifetimes = dgm[:, 1] - dgm[:, 0]
            rows.append((len(dgm), float(lifetimes.sum()), float(lifetimes.max())))
    return np.array(rows, dtype=np.float32)

def compute_diagrams(images: torch.Tensor) -> list[np.ndarray]:
    """H1 diagram for each image in (N, 1, 28, 28)."""
    arr = images.detach().cpu().squeeze(1).numpy()
    return [_h1_diagram(img) for img in arr]


def compute_reference_diagrams(
    real: torch.Tensor,
    n_samples: int = 200,
    seed: int = 0,
) -> list[np.ndarray]:
    """H1 diagrams for a random subset of real images — used as the
    reference distribution for topological novelty."""
    rng = np.random.default_rng(seed)
    n = min(n_samples, len(real))
    idx = rng.choice(len(real), size=n, replace=False)
    return compute_diagrams(real[idx])

class ManifoldKNN:
    """Hypersphere-union manifold estimate of Kynkaanniemi et al. 2019.

    For every real feature vector r_i we define a hypersphere of radius
    R_i = ||r_i - NN_k(r_i)||, where NN_k is its k-th nearest neighbour
    among the real features.  A query point q is "on the manifold" iff
    there exists some i with ||q - r_i|| <= R_i.
    """

    def __init__(self, real_features: np.ndarray, k: int = 3):
        self.real = np.asarray(real_features, dtype=np.float32)
        self.k = k
        nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(self.real)
        dists, _ = nbrs.kneighbors(self.real)
        self.radii = dists[:, k].astype(np.float32)   # (N_real,)
        self._query_nbrs = NearestNeighbors(metric="euclidean").fit(self.real)

    def in_manifold(self, query_features: np.ndarray) -> np.ndarray:
        """Binary membership: 1 if inside the union of hyperspheres."""
        q = np.asarray(query_features, dtype=np.float32)
        # Only need to check the real samples whose hyperspheres could
        # possibly contain q; the largest radius bounds the search.
        r_max = float(self.radii.max())
        nbrs = self._query_nbrs.radius_neighbors(q, radius=r_max,
                                                 return_distance=True)
        dists, idx = nbrs
        out = np.zeros(len(q), dtype=np.float32)
        for i, (d, j) in enumerate(zip(dists, idx)):
            if len(d) == 0:
                continue
            if np.any(d <= self.radii[j]):
                out[i] = 1.0
        return out

    def soft_realness(self, query_features: np.ndarray) -> np.ndarray:
        """Continuous realness in (0, 1].

        For each query, take the real sample r* that minimises
        rho = ||q - r|| / R(r) and report 1 / (1 + rho).  rho == 0 means
        the query sits on top of a real sample (score 1), rho == 1 puts it
        exactly on some hypersphere surface (score 0.5), and large rho
        decays toward 0.  Unlike the paper's binary membership test this
        keeps differentiating signal *inside* the manifold — the original
        clip-to-1 version saturated for any decoder output that landed in
        some hypersphere, killing the RL reward gradient."""
        q = np.asarray(query_features, dtype=np.float32)
        dists, idx = self._query_nbrs.kneighbors(q, n_neighbors=len(self.real))
        ratios = dists / np.maximum(self.radii[idx], 1e-12)     # (N_q, N_real)
        best = ratios.min(axis=1)
        return (1.0 / (1.0 + best)).astype(np.float32)

def _coerce_images(x) -> torch.Tensor:
    """Normalise a batch of 28x28 images to an (N, 1, 28, 28) float tensor
    in [0, 1].  Accepts: torch.Tensor, np.ndarray, a single PIL-loadable
    image path, or a list/tuple of such paths."""
    # List of paths → stack after loading.
    if isinstance(x, (list, tuple)) and len(x) > 0 and isinstance(x[0], (str, os.PathLike)):
        from PIL import Image
        arrs = []
        for p in x:
            im = Image.open(p).convert("L")
            if im.size != (28, 28):
                im = im.resize((28, 28))
            arrs.append(np.asarray(im, dtype=np.float32) / 255.0)
        t = torch.from_numpy(np.stack(arrs))
    elif isinstance(x, (str, os.PathLike)):
        return _coerce_images([x])
    elif isinstance(x, np.ndarray):
        t = torch.from_numpy(x.astype(np.float32, copy=False))
    elif isinstance(x, torch.Tensor):
        t = x.detach().cpu().float()
    else:
        raise TypeError(f"unsupported image input type: {type(x).__name__}")

    if t.dim() == 2:       # single (28, 28)
        t = t.unsqueeze(0).unsqueeze(0)
    elif t.dim() == 3:     # (N, 28, 28)
        t = t.unsqueeze(1)
    elif t.dim() != 4:
        raise ValueError(f"expected 2/3/4-D image tensor, got shape {tuple(t.shape)}")

    if t.shape[-1] != 28 or t.shape[-2] != 28:
        raise ValueError(f"expected 28x28 images, got {tuple(t.shape)}")
    if float(t.max()) > 1.5:      # heuristic: values look like 0-255
        t = t / 255.0
    return t.clamp(0.0, 1.0)

def pixel_features(images: torch.Tensor) -> np.ndarray:
    """Flatten (N, 1, 28, 28) → (N, 784)."""
    return images.reshape(images.size(0), -1).detach().cpu().numpy()

def load_mnist_2_and_6(root: str = "./data", train: bool = True) -> torch.Tensor:
    """Return the subset of MNIST restricted to digits 2 and 6 as an
    (N, 1, 28, 28) float tensor in [0, 1]."""
    ds = torchvision.datasets.MNIST(
        root=root, train=train, download=True,
        transform=transforms.ToTensor(),
    )
    targets = ds.targets if isinstance(ds.targets, torch.Tensor) else torch.tensor(ds.targets)
    mask = (targets == 2) | (targets == 6)
    idx = torch.nonzero(mask, as_tuple=False).flatten().tolist()
    subset = Subset(ds, idx)
    loader = DataLoader(subset, batch_size=512, shuffle=False)
    xs = [x for x, _ in loader]
    return torch.cat(xs, dim=0)

def evaluate_pregenerated(
    images,
    real: Optional[torch.Tensor] = None,
    *,
    k_manifold: int = 3,
    alpha: float = 0.5,
    realness_mode: str = "soft",
    n_reference_diagrams: int = 150,
    feature_fn: Optional[FeatureFn] = None,
    manifold: Optional[ManifoldKNN] = None,
    reference_diagrams: Optional[Sequence[np.ndarray]] = None,
    return_images: bool = False,
) -> dict:
    """Score creativity for images generated elsewhere.

    `images` may be a torch.Tensor, a numpy array (shape (N,28,28),
    (N,1,28,28), or (28,28)), a single path, or a list of paths to
    PNG/JPG files.  Values outside [0, 1] are auto-scaled from 0–255.

    `real` is the reference pool (e.g. `load_mnist_2_and_6()`).  If you
    are scoring many batches, pass `manifold` and `reference_diagrams`
    from a previous call instead — the K-2019 fit and the reference H1
    diagrams are the expensive parts and are fully reusable.

    Returns a dict with per-image arrays
    (creativity, realness, h1_novelty, h1_summary) plus the cached
    `manifold` and `reference_diagrams` so the caller can reuse them."""
    imgs = _coerce_images(images)

    ff = feature_fn if feature_fn is not None else pixel_features
    if manifold is None:
        if real is None:
            raise ValueError("pass `real` or a cached `manifold`")
        manifold = ManifoldKNN(ff(real), k=k_manifold)

    gen_feat = ff(imgs)
    if realness_mode == "binary":
        realness = manifold.in_manifold(gen_feat)
    elif realness_mode == "soft":
        realness = manifold.soft_realness(gen_feat)
    else:
        raise ValueError(f"unknown realness_mode {realness_mode!r}")

    if _TOPO_OK:
        if reference_diagrams is None:
            if real is None:
                raise ValueError("pass `real` or cached `reference_diagrams`")
            reference_diagrams = compute_reference_diagrams(
                real, n_samples=n_reference_diagrams,
            )
        gen_diagrams = compute_diagrams(imgs)
        novelty = h1_novelty(gen_diagrams, reference_diagrams)
        summary = h1_summary(gen_diagrams)
    else:
        reference_diagrams = reference_diagrams or []
        novelty = np.zeros(len(imgs), dtype=np.float32)
        summary = np.zeros((len(imgs), 3), dtype=np.float32)

    creativity = (realness ** alpha) * (novelty ** (1.0 - alpha))
    out = {
        "creativity":         creativity,
        "realness":           realness,
        "h1_novelty":         novelty,
        "h1_summary":         summary,
        "alpha":              alpha,
        "manifold":           manifold,
        "reference_diagrams": reference_diagrams,
    }
    if return_images:
        out["images"] = imgs
    return out

def plot_creativity_landscape(
    realness: np.ndarray,
    novelty: np.ndarray,
    creativity: Optional[np.ndarray] = None,
    *,
    alpha: float = 0.5,
    labels: Optional[Sequence[str]] = None,
    images: Optional[torch.Tensor] = None,
    thumbnail_zoom: float = 0.45,
    annotate_top_k: Optional[int] = 5,
    title: str = "Creativity landscape",
    ax=None,
):
    """Plot the Creativity = Realness^alpha · Novelty^(1-alpha) landscape.

    Background is the iso-creativity heatmap on the unit square; the
    evaluated images are overlaid as points (or thumbnails) located at
    their (realness, novelty) coordinates and coloured by creativity.

    Parameters
    ----------
    realness, novelty    per-image scores in [0, 1].
    creativity           optional; recomputed from alpha if omitted.
    alpha                mixing weight used to draw the background.
    labels               short per-image strings for the top-k annotations.
    images               (N, 1, 28, 28) tensor — if provided, images are
                         drawn as thumbnails instead of scatter markers.
    annotate_top_k       how many highest-creativity points to label.
    """
    import matplotlib.pyplot as plt

    viridis = plt.get_cmap("viridis")

    realness   = np.asarray(realness,   dtype=np.float32)
    novelty    = np.asarray(novelty,    dtype=np.float32)
    if creativity is None:
        creativity = (realness ** alpha) * (novelty ** (1.0 - alpha))
    creativity = np.asarray(creativity, dtype=np.float32)

    if ax is None:
        _, ax = plt.subplots(figsize=(6.5, 5.5))

    grid = np.linspace(0.0, 1.0, 200)
    R, N = np.meshgrid(grid, grid)
    C = (R ** alpha) * (N ** (1.0 - alpha))

    im = ax.imshow(
        C, origin="lower", extent=(0, 1, 0, 1), cmap="viridis",
        vmin=0.0, vmax=1.0, alpha=0.85, aspect="equal",
    )
    cs = ax.contour(
        R, N, C, levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
        colors="white", linewidths=0.6, alpha=0.6,
    )
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.1f")

    if images is not None:
        from matplotlib.offsetbox import OffsetImage, AnnotationBbox
        imgs_np = images.detach().cpu().numpy()
        if imgs_np.ndim == 4:
            imgs_np = imgs_np[:, 0]
        order = np.argsort(creativity)          # draw best on top
        for i in order:
            ab = AnnotationBbox(
                OffsetImage(imgs_np[i], cmap="gray", zoom=thumbnail_zoom),
                (float(realness[i]), float(novelty[i])),
                frameon=True, pad=0.15,
                bboxprops=dict(
                    edgecolor=viridis(float(creativity[i])),
                    linewidth=1.4,
                ),
            )
            ax.add_artist(ab)
    else:
        ax.scatter(
            realness, novelty, c=creativity, cmap="viridis",
            vmin=0.0, vmax=1.0, edgecolors="white", linewidths=0.6, s=55,
        )

    if annotate_top_k and labels is not None:
        top = np.argsort(-creativity)[: int(annotate_top_k)]
        for i in top:
            ax.annotate(
                labels[i],
                xy=(float(realness[i]), float(novelty[i])),
                xytext=(6, 6), textcoords="offset points",
                fontsize=8, color="white",
                bbox=dict(facecolor="black", alpha=0.55,
                          edgecolor="none", pad=1.2),
            )

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("Realness  (K-2019 manifold)")
    ax.set_ylabel("H1 novelty  (persistence distance)")
    ax.set_title(f"{title}   (α = {alpha:.2f})")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Creativity")

    return ax

def _eval_pregenerated_demo(
    folder: str = "./images",
    exts: Sequence[str] = ("png", "jpg", "jpeg", "bmp"),
    top_k_to_plot: Optional[int] = None,
) -> None:
    """Score every image in `folder` with the K-2019 × H1 creativity
    metric and print + plot a ranked comparison.  `_coerce_images`
    handles resizing to 28x28 and 0-255 → 0-1 scaling automatically."""
    import glob
    import matplotlib.pyplot as plt

    paths: list[str] = []
    for ext in exts:
        paths += glob.glob(os.path.join(folder, f"*.{ext}"))
        paths += glob.glob(os.path.join(folder, f"*.{ext.upper()}"))
    paths = sorted(set(paths))
    if not paths:
        print(f"No images found in {folder!r}")
        return
    print(f"Scoring {len(paths)} image(s) from {folder} ...")

    real = load_mnist_2_and_6(train=True)
    out = evaluate_pregenerated(
        paths, real, n_reference_diagrams=120, return_images=True,
    )

    order = np.argsort(-out["creativity"])   # descending
    creat, realn, nov = out["creativity"], out["realness"], out["h1_novelty"]
    summ = out["h1_summary"]

    print(f"\n{'rank':>4}  {'creat':>6}  {'real':>6}  {'H1nov':>6}  "
          f"{'H1feat':>6}  file")
    for rank, idx in enumerate(order):
        print(f"{rank+1:>4}  {creat[idx]:>6.3f}  {realn[idx]:>6.3f}  "
              f"{nov[idx]:>6.3f}  {int(summ[idx, 0]):>6d}  "
              f"{os.path.basename(paths[idx])}")

    n_plot = len(paths) if top_k_to_plot is None else min(top_k_to_plot, len(paths))
    cols = min(n_plot, 6)
    rows = (n_plot + cols - 1) // cols
    _, axes = plt.subplots(rows, cols, figsize=(2.0 * cols, 2.3 * rows),
                           squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for plot_idx in range(n_plot):
        idx = int(order[plot_idx])
        r, c = divmod(plot_idx, cols)
        ax = axes[r][c]
        ax.imshow(out["images"][idx].squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
        name = os.path.basename(paths[idx])
        if len(name) > 18:
            name = name[:15] + "…"
        ax.set_title(
            f"{name}\nc={creat[idx]:.2f} r={realn[idx]:.2f} n={nov[idx]:.2f}",
            fontsize=8,
        )
    plt.suptitle(f"Creativity of images in {folder}  (ranked)", fontsize=10)
    plt.tight_layout()
    plt.show()

    labels = [os.path.splitext(os.path.basename(p))[0] for p in paths]
    plot_creativity_landscape(
        realn, nov, creat,
        alpha=float(out.get("alpha", 0.5)),
        labels=labels,
        images=out["images"],
        title=f"Creativity landscape — {folder}",
    )
    plt.tight_layout()
    plt.show()
